In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd

# 加载训练好的模型
model_path = "my_final_model/my_model_2"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("✅ 模型加载成功！")

# 情绪映射（与训练时一致）
emotion_mapping = {
    0: "开心", 1: "难过", 2: "愤怒", 3: "恐惧", 
    4: "惊讶", 5: "平静", 6: "厌恶", 7: "困惑"
}

def predict_emotion(text):
    """预测单条文本的情绪"""
    # 编码文本
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True, 
        max_length=128
    )
    
    # 模型预测
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # 获取所有情绪的概率
    probabilities = predictions[0].numpy()
    
    # 找到主要情绪
    predicted_class = probabilities.argmax()
    confidence = probabilities[predicted_class]
    emotion = emotion_mapping[predicted_class]
    
    # 获取所有情绪的概率（按置信度排序）
    emotion_probs = []
    for i, prob in enumerate(probabilities):
        emotion_probs.append({
            'emotion': emotion_mapping[i],
            'confidence': prob
        })
    
    emotion_probs.sort(key=lambda x: x['confidence'], reverse=True)
    
    return {
        'text': text,
        'primary_emotion': emotion,
        'primary_confidence': confidence,
        'all_emotions': emotion_probs
    }

# 测试各种情绪
test_texts = [
    # 开心 - 积极事件
    "今天阳光特别好，窗外的鸟儿在唱歌",
    "刚刚完成了一个大项目，感觉轻松了很多",
    "朋友突然来访，还带了小礼物",
    
    # 难过 - 失落事件  
    "雨一直下，房间里显得格外安静",
    "翻看旧照片，想起了很多往事",
    "精心准备的东西没有被注意到",
    
    # 愤怒 - 不公平事件
    "明明提前到了，位置却被别人占了",
    "答应好的事情临时变卦，没有任何解释",
    "排队时有人直接插到前面",
    
    # 恐惧 - 不安事件
    "深夜独自回家，路灯忽明忽暗",
    "听到门外有奇怪的声响，但看不清是什么",
    "站在高处往下看，腿有些发软",
    
    # 惊讶 - 意外事件
    "打开门发现客厅布置得完全不一样",
    "原本以为不会来的人突然出现",
    "考试成绩比预想的要好很多",
    
    # 平静 - 日常事件
    "泡了一杯茶，坐在窗边看书",
    "周末没有什么安排，可以好好休息",
    "完成工作后，时间还早",
    
    # 厌恶 - 不适事件
    "食物里有奇怪的味道，难以下咽",
    "公共场所看到不文明的行为",
    "闻到一股刺鼻的气味",
    
    # 困惑 - 不解事件
    "这个操作步骤看了几遍还是不太明白",
    "对方说的话前后矛盾，不知道哪个是真的",
    "面对多个选择，每个都有利弊"
]


print("🧪 情绪分类测试结果:")
print("=" * 70)

for text in test_texts:
    result = predict_emotion(text)
    
    print(f"📝 文本: {result['text']}")
    print(f"🎯 主要情绪: {result['primary_emotion']} (置信度: {result['primary_confidence']:.2%})")
    
    print("📊 详细概率:")
    for i, emotion_info in enumerate(result['all_emotions'][:3]):  # 显示前3个
        print(f"   {i+1}. {emotion_info['emotion']}: {emotion_info['confidence']:.2%}")
    
    print("-" * 70)

✅ 模型加载成功！
🧪 情绪分类测试结果:
📝 文本: 今天阳光特别好，窗外的鸟儿在唱歌
🎯 主要情绪: 开心 (置信度: 66.34%)
📊 详细概率:
   1. 开心: 66.34%
   2. 难过: 16.61%
   3. 平静: 13.85%
----------------------------------------------------------------------
📝 文本: 刚刚完成了一个大项目，感觉轻松了很多
🎯 主要情绪: 开心 (置信度: 97.98%)
📊 详细概率:
   1. 开心: 97.98%
   2. 平静: 0.55%
   3. 难过: 0.45%
----------------------------------------------------------------------
📝 文本: 朋友突然来访，还带了小礼物
🎯 主要情绪: 开心 (置信度: 43.29%)
📊 详细概率:
   1. 开心: 43.29%
   2. 平静: 33.75%
   3. 难过: 18.18%
----------------------------------------------------------------------
📝 文本: 雨一直下，房间里显得格外安静
🎯 主要情绪: 难过 (置信度: 51.05%)
📊 详细概率:
   1. 难过: 51.05%
   2. 平静: 46.15%
   3. 开心: 0.94%
----------------------------------------------------------------------
📝 文本: 翻看旧照片，想起了很多往事
🎯 主要情绪: 难过 (置信度: 99.18%)
📊 详细概率:
   1. 难过: 99.18%
   2. 平静: 0.23%
   3. 恐惧: 0.18%
----------------------------------------------------------------------
📝 文本: 精心准备的东西没有被注意到
🎯 主要情绪: 难过 (置信度: 98.00%)
📊 详细概率:
   1. 难过: 98.00%
   2. 平静: 0.69%
   3. 开心: 0.4

In [3]:
import torch
import psutil
import GPUtil

print("=== 系统资源检查 ===")

# 检查GPU
if torch.cuda.is_available():
    print(f"✅ CUDA可用")
    print(f"🖥️  GPU设备数量: {torch.cuda.device_count()}")
    
    for i in range(torch.cuda.device_count()):
        gpu_props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {gpu_props.name}")
        print(f"    显存: {gpu_props.total_memory / 1024**3:.1f} GB")
        print(f"    CUDA核心: {gpu_props.multi_processor_count}")
else:
    print("❌ CUDA不可用")

# 检查当前设备
current_device = torch.cuda.current_device() if torch.cuda.is_available() else None
print(f"📱 当前设备: {current_device}")

# 检查模型是否在GPU上
print(f"🧠 模型设备: {next(model.parameters()).device}")

# 使用GPUtil检查更详细的GPU信息
try:
    gpus = GPUtil.getGPUs()
    for gpu in gpus:
        print(f"🎯 GPU {gpu.id}: {gpu.name}")
        print(f"   显存使用: {gpu.memoryUsed}MB / {gpu.memoryTotal}MB")
        print(f"   显存利用率: {gpu.memoryUtil*100:.1f}%")
        print(f"   GPU利用率: {gpu.load*100:.1f}%")
except:
    print("⚠️  GPUtil未安装，使用 'pip install gputil' 安装")

=== 系统资源检查 ===
✅ CUDA可用
🖥️  GPU设备数量: 1
  GPU 0: NVIDIA GeForce RTX 5060 Laptop GPU
    显存: 8.0 GB
    CUDA核心: 26
📱 当前设备: 0


NameError: name 'model' is not defined

In [2]:
# check_cuda.py
import torch
import subprocess
import sys

print("=== 开始CUDA环境检查 ===\n")

print("1. 检查PyTorch版本和内置CUDA支持:")
print(f"   PyTorch版本: {torch.__version__}")
print(f"   PyTorch内置CUDA支持: {torch.cuda.is_available()}")
print(f"   PyTorch编译时的CUDA版本: {torch.version.cuda}\n")

print("2. 尝试检查CUDA设备:")
if torch.cuda.is_available():
    print(f"   检测到的GPU设备数量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   设备 {i}: {torch.cuda.get_device_name(i)}")
else:
    print("   ❌ PyTorch无法检测到CUDA设备。\n")

print("3. 系统环境信息:")
# 检查环境变量
cuda_path = sys.exec_prefix
print(f"   Python环境路径: {cuda_path}")

print("\n=== 检查结束 ===")

=== 开始CUDA环境检查 ===

1. 检查PyTorch版本和内置CUDA支持:
   PyTorch版本: 2.8.0+cu129
   PyTorch内置CUDA支持: True
   PyTorch编译时的CUDA版本: 12.9

2. 尝试检查CUDA设备:
   检测到的GPU设备数量: 1
   设备 0: NVIDIA GeForce RTX 5060 Laptop GPU
3. 系统环境信息:
   Python环境路径: D:\anaconda3\envs\model_nlp

=== 检查结束 ===
